In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# --- تنظیمات آدرس‌ها ---
file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
output_filename = r'outputs\G11\dsas_g11_bearings_vibration_temp_clustering\clustering\dsas_g11_clustering_output4.xlsx'

# سنسورهایی که به عنوان فیچر در کلاسترینگ استفاده می‌شوند
target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

def run_hybrid_analysis():
    if not os.path.exists(file_path):
        print(f"❌ فایل یافت نشد: {file_path}")
        return

    # ۱. خواندن داده‌ها
    df = pd.read_excel(file_path)
    
    # بررسی موجود بودن ستون‌ها در فایل شما
    available_sensors = [col for col in target_sensors if col in df.columns]
    
    if not available_sensors:
        print("❌ هیچ‌کدام از سنسورهای هدف در فایل یافت نشدند.")
        print(f"ستون‌های موجود در فایل شما: {df.columns.tolist()}")
        return

    # ۲. آماده‌سازی داده‌ها (حذف ردیف‌های خالی)
    df_model = df.dropna(subset=available_sensors).copy()

    # ۳. استانداردسازی (بسیار مهم برای کلاسترینگ)
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_model[available_sensors])

    # ۴. بخش اول: Anomaly Detection (با Isolation Forest)
    iso_forest = IsolationForest(contamination=0.02, random_state=42)
    df_model['Anomaly_Score'] = iso_forest.fit_predict(scaled_data)
    df_model['Is_Anomaly'] = df_model['Anomaly_Score'].apply(lambda x: 'Yes' if x == -1 else 'No')

    # ۵. بخش دوم: Clustering (با K-Means برای شناسایی وضعیت‌ها)
    # ۳ خوشه: احتمالاً وضعیت نرمال، وضعیت گذرا (Warning) و وضعیت تغییر رفتار شدید
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    df_model['System_State_Cluster'] = kmeans.fit_predict(scaled_data)

    # ۶. ترکیب خروجی (این ستون همان برچسبی است که برای کلاسیفیکیشن استفاده می‌شود)
    # ما یک وضعیت "تارگت" ترکیبی می‌سازیم
    df_model['Final_Health_Label'] = df_model.apply(
        lambda row: 'Critical' if row['Is_Anomaly'] == 'Yes' else f'State_{row["System_State_Cluster"]}', 
        axis=1
    )

    # ۷. ذخیره در اکسل
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        df_model.to_excel(output_filename, index=False)
        print(f"🚀 تحلیل ترکیبی با موفقیت انجام شد!")
        print(f"✅ آنومالی‌ها شناسایی شدند و داده‌ها به ۳ وضعیت سیستمی کلاستر شدند.")
        print(f"📁 فایل خروجی: {output_filename}")
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")

if __name__ == "__main__":
    run_hybrid_analysis()


c:\Users\pishva_r\.conda\envs\base2\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


🚀 تحلیل ترکیبی با موفقیت انجام شد!
✅ آنومالی‌ها شناسایی شدند و داده‌ها به ۳ وضعیت سیستمی کلاستر شدند.
📁 فایل خروجی: outputs\G11\dsas_g11_clustering_output_hybrid.xlsx


In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# --- تنظیمات آدرس‌ها ---
file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
output_filename = r'outputs\G11\smart_engineering_analysis.xlsx'

target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

def run_smart_analysis():
    if not os.path.exists(file_path): return

    df = pd.read_excel(file_path)
    df_model = df.dropna(subset=target_sensors).copy()

    # گام طلایی ۱: حذف نویزهای لحظه‌ای (Smoothing)
    # این کار باعث می‌شود پرش‌های اشتباه اپراتور حذف شود و فقط "تغییرات واقعی" باقی بماند
    for sensor in target_sensors:
        df_model[f'{sensor}_smooth'] = df_model[sensor].rolling(window=5, center=True).mean()

    df_model = df_model.dropna() # حذف ردیف‌های لبه به دلیل rolling
    smooth_cols = [f'{s}_smooth' for s in target_sensors]

    # گام طلایی ۲: محاسبه انحراف از رفتار نرمال (Z-Score)
    # ما نمی‌گوییم داده پرت است، می‌گوییم چقدر از "خودِ همیشگی‌اش" فاصله گرفته
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_model[smooth_cols])

    # گام ۳: کلاسترینگ بر اساس رفتار پایدار
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    df_model['Behavior_Cluster'] = kmeans.fit_predict(scaled_data)


    # گام ۴: شاخص سلامت مهندسی (Engineering Health Index)
    # فاصله اقلیدسی از مرکز خوشه "نرمال" (بجای برچسب آنومالی)
    distances = np.linalg.norm(scaled_data - kmeans.cluster_centers_[df_model['Behavior_Cluster']], axis=1)
    df_model['Degradation_Index'] = distances # شاخص تخریب (هر چه بیشتر، یعنی یاتاقان دارد پیر می‌شود)

    # گام ۵: برچسب‌گذاری مدیریتی
    def manage_label(row):
        if row['Degradation_Index'] > 2.5: # انحراف زیاد اما واقعی (چون دیتا صاف شده)
            return "نیاز به بازبینی فنی (تغییر رفتار)"
        elif row['Degradation_Index'] > 1.5:
            return "عملکرد تحت مراقبت"
        else:
            return "عملکرد بهینه"

    df_model['Management_Report'] = df_model.apply(manage_label, axis=1)

    # ذخیره سازی
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        df_model.to_excel(output_filename, index=False)
        print("🚀 مدل مهندسی اجرا شد. نویزهای لحظه‌ای حذف و تغییرات رفتاری شناسایی شدند.")
    except Exception as e:
        print(f"❌ خطا: {e}")

if name == "__main__":
    run_smart_analysis()